In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — leave defaults for a full unattended run.
from pathlib import Path
RUN_PLAQUE_ENSEMBLE=True
RUN_CORONARY_ENSEMBLE=True
RUN_WHOLE_HEART=True
KEEP_GOING_ON_ERROR=True
REUSE_PLAQUE_PREDICTIONS=True
REUSE_CORONARY_CURRENT=True
REUSE_CORONARY_LEGACY=True
REUSE_HEART_TOTAL=True
REUSE_AORTIC_SINUSES=True
SAVE_CORONARY_PROBABILITIES=True
RUN_HIGHRES_CHAMBERS_IF_LICENSED=True
LICENSE_FILE=Path('/content/drive/MyDrive/OpenPlaque/private/totalseg_license.txt')
TOTALSEG_LICENSE = LICENSE_FILE.read_text().strip() if LICENSE_FILE.exists() else ''
print('TotalSegmentator Drive license file found:', bool(TOTALSEG_LICENSE))


# OpenPlaque — unattended GPU batch

Runs **plaque 5-fold ensemble → coronary ensemble → whole-heart context** sequentially. Expensive outputs are cached to Drive and reused after interruptions. The TotalSegmentator license is read from a private Google Drive text file and is never written into GitHub, reports, command logs, or notebook output.

Expected license file: `/content/drive/MyDrive/OpenPlaque/private/totalseg_license.txt`

In [ ]:
!pip -q install nnunetv2 TotalSegmentator SimpleITK pydicom pandas matplotlib nibabel

import os, sys, shutil, subprocess
from pathlib import Path

repo=Path('/content/OpenPlaque')
if repo.exists():
    shutil.rmtree(repo)
!git clone -q --depth 1 --branch gpu-batch-pipeline-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0,'/content/OpenPlaque/src')

!nvidia-smi

if TOTALSEG_LICENSE:
    subprocess.run(['totalseg_set_license','-l',TOTALSEG_LICENSE], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    print('TotalSegmentator license activated for this runtime.')
else:
    print('No Drive license file found; licensed tasks will be skipped.')


In [ ]:
# Import the batch runner and ensure the activated local license state is used
# without placing the license on any logged TotalSegmentator command line.
import shutil, subprocess
from pathlib import Path
import openplaque.gpu_batch as gb

def _totalseg_activated(source, out, task, reuse=True, probabilities=False, license_number='', preview=False):
    out=Path(out)
    done=out/'_SUCCESS'
    if reuse and done.exists():
        print('reuse', task, out)
        return out
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    cmd=['TotalSegmentator','-i',str(source),'-o',str(out),'-ta',task,'--device','gpu']
    if preview:
        cmd += ['--preview']
    if probabilities:
        cmd += ['--save_probabilities',str(out/f'{out.name}_probabilities.npz')]
    try:
        gb._run(cmd)
    except subprocess.CalledProcessError:
        if probabilities:
            print('Probability output failed; retrying mask-only.')
            shutil.rmtree(out)
            out.mkdir(parents=True)
            gb._run(['TotalSegmentator','-i',str(source),'-o',str(out),'-ta',task,'--device','gpu'])
        else:
            raise
    done.write_text('done')
    return out

gb._totalseg = _totalseg_activated
print('Licensed TotalSegmentator tasks will use activated local license state.')


In [ ]:
BATCH_ROOT='/content/drive/MyDrive/OpenPlaque/GPU_Batch_Pipeline_v1'
cfg=dict(
 RUN_PLAQUE_ENSEMBLE=RUN_PLAQUE_ENSEMBLE, RUN_CORONARY_ENSEMBLE=RUN_CORONARY_ENSEMBLE, RUN_WHOLE_HEART=RUN_WHOLE_HEART, KEEP_GOING_ON_ERROR=KEEP_GOING_ON_ERROR,
 REUSE_PLAQUE_PREDICTIONS=REUSE_PLAQUE_PREDICTIONS, REUSE_CORONARY_CURRENT=REUSE_CORONARY_CURRENT, REUSE_CORONARY_LEGACY=REUSE_CORONARY_LEGACY, REUSE_HEART_TOTAL=REUSE_HEART_TOTAL, REUSE_AORTIC_SINUSES=REUSE_AORTIC_SINUSES,
 SAVE_CORONARY_PROBABILITIES=SAVE_CORONARY_PROBABILITIES, RUN_HIGHRES_CHAMBERS_IF_LICENSED=RUN_HIGHRES_CHAMBERS_IF_LICENSED,
 TOTALSEG_LICENSE=('ACTIVE' if TOTALSEG_LICENSE else ''),
 FOLDS=[0,1,2,3,4], VESSELS=['RCA','LAD'], SERIES={'RCA':1035,'LAD':1043},
 MODEL_ZIP='/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip',
 STUDY_ZIP='/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip', SOURCE_CACHE='/content/drive/MyDrive/OpenPlaque/Cache/Secondary_3D_Vesselness_Topology_v1',
 BATCH_ROOT=BATCH_ROOT, PLAQUE_ROOT=BATCH_ROOT+'/01_plaque_5fold', CORONARY_ROOT=BATCH_ROOT+'/02_coronary_ensemble', HEART_ROOT=BATCH_ROOT+'/03_whole_heart')
status, master_zip = gb.run_all(cfg)
print(status)
print('MASTER ZIP:', master_zip)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_GPU_BATCH_REPORT_BACK.zip')
